# Option 2: Causal Forests
**Replication**: Wager & Athey (2018) - Estimation and Inference of Heterogeneous Treatment Effects
**Data**: NSW Job Training Program (LaLonde, 1986; Dehejia & Wahba, 1999)

**Key Question**: Which types of workers benefit most from job training?

## 1. Setup and Data Loading

In [ ]:
# Download data files if not already present
import os
import urllib.request

BASE_URL = "https://raw.githubusercontent.com/JasmineHao/JasmineHao.github.io/main/econ6083/final-project/notebooks/data/"
DATA_FILES = ['nsw_mixtape.csv']

os.makedirs('data', exist_ok=True)
for fname in DATA_FILES:
    if not os.path.exists(f'data/{fname}'):
        print(f"Downloading {fname} ...")
        urllib.request.urlretrieve(BASE_URL + fname, f'data/{fname}')
        print(f"  Saved to data/{fname}")
    else:
        print(f"Found local: data/{fname}")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Download NSW data from local file
# Dehejia & Wahba (1999) - widely used for causal forest replication
df = pd.read_csv('data/nsw_mixtape.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

## 2. Data Exploration

In [ ]:
# Key variables
# re78: Real earnings in 1978 (outcome)
# treat: Treatment assignment (1 = NSW job training)
# covariates: age, educ, black, hisp, marr, nodegree, re75, re74

print("Outcome: Real Earnings 1978")
print(df['re78'].describe())

print("\nTreatment Assignment")
print(df['treat'].value_counts())

print("\nATE (Simple Difference in Means):")
ate_simple = df[df['treat']==1]['re78'].mean() - df[df['treat']==0]['re78'].mean()
print(f"ATE = ${ate_simple:.2f}")

print("\nCovariate balance:")
covs = ['age', 'educ', 'black', 'hisp', 'marr', 'nodegree', 're74', 're75']
print(df.groupby('treat')[covs].mean().round(2))

## 3. Causal Tree Implementation

In [ ]:
class CausalTree:
    """
    Simplified Causal Tree implementation following Athey & Imbens (2016)
    """

    def __init__(self, max_depth=3, min_samples_leaf=20):
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf

    def fit(self, X, Y, D):
        """Fit causal tree by recursively partitioning"""
        self.tree = self._grow_tree(X, Y, D, depth=0)
        return self

    def _grow_tree(self, X, Y, D, depth):
        """Recursively grow tree"""
        n = len(Y)

        if depth >= self.max_depth or n < 2 * self.min_samples_leaf:
            return self._create_leaf(Y, D)

        best_gain = -np.inf
        best_split = None

        n_features = X.shape[1]
        for feature in range(n_features):
            thresholds = np.percentile(X[:, feature], [25, 50, 75])
            for threshold in thresholds:
                left = X[:, feature] <= threshold
                right = ~left

                if left.sum() < self.min_samples_leaf or right.sum() < self.min_samples_leaf:
                    continue

                tau_left = self._treatment_effect(Y[left], D[left])
                tau_right = self._treatment_effect(Y[right], D[right])
                gain = abs(tau_left - tau_right)

                if gain > best_gain:
                    best_gain = gain
                    best_split = {'feature': feature, 'threshold': threshold}

        if best_split is None:
            return self._create_leaf(Y, D)

        left_idx = X[:, best_split['feature']] <= best_split['threshold']
        right_idx = ~left_idx

        return {
            'feature': best_split['feature'],
            'threshold': best_split['threshold'],
            'left': self._grow_tree(X[left_idx], Y[left_idx], D[left_idx], depth+1),
            'right': self._grow_tree(X[right_idx], Y[right_idx], D[right_idx], depth+1)
        }

    def _treatment_effect(self, Y, D):
        if D.sum() == 0 or (D==0).sum() == 0:
            return 0
        return Y[D==1].mean() - Y[D==0].mean()

    def _create_leaf(self, Y, D):
        return {'tau': self._treatment_effect(Y, D), 'is_leaf': True}

    def predict(self, X):
        return np.array([self._predict_one(x, self.tree) for x in X])

    def _predict_one(self, x, tree):
        if tree.get('is_leaf'):
            return tree['tau']
        if x[tree['feature']] <= tree['threshold']:
            return self._predict_one(x, tree['left'])
        else:
            return self._predict_one(x, tree['right'])

## 4. Prepare Data

In [ ]:
# Define variables
Y = df['re78'].values
D = df['treat'].values

# Covariates
covariates = ['age', 'educ', 'black', 'hisp', 'marr', 'nodegree', 're74', 're75']
X = df[covariates].values

# Train-test split for honest estimation
X_train, X_test, Y_train, Y_test, D_train, D_test = train_test_split(
    X, Y, D, test_size=0.3, random_state=42
)

# Save indices for subgroup analysis later
train_indices, test_indices = train_test_split(
    np.arange(len(df)), test_size=0.3, random_state=42
)

print(f"Train size: {len(Y_train)}, Test size: {len(Y_test)}")
print(f"Treatment rate in train: {D_train.mean():.3f}")

## 5. Fit Causal Tree and Analyze Heterogeneity

In [ ]:
# Fit causal tree
ct = CausalTree(max_depth=3, min_samples_leaf=15)
ct.fit(X_train, Y_train, D_train)

# Predict CATE on test set
cate_pred = ct.predict(X_test)

print("CATE Distribution on Test Set:")
print(f"Mean CATE: {cate_pred.mean():.2f}")
print(f"Std CATE: {cate_pred.std():.2f}")
print(f"Min CATE: {cate_pred.min():.2f}")
print(f"Max CATE: {cate_pred.max():.2f}")

# Plot CATE distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.hist(cate_pred, bins=30, edgecolor='black', alpha=0.7)
ax1.axvline(x=0, color='red', linestyle='--')
ax1.set_xlabel('CATE ($)')
ax1.set_ylabel('Count')
ax1.set_title('Distribution of CATE Estimates')
ax1.grid(alpha=0.3)

ax2 = axes[1]
# Sort by predicted CATE for visualization
sort_idx = np.argsort(cate_pred)
x_pos = np.arange(len(sort_idx))
ax2.scatter(x_pos, Y_test[sort_idx], alpha=0.3, label='Observed Outcome', s=20)
ax2.plot(x_pos, cate_pred[sort_idx], color='red', linewidth=2, label='Predicted CATE')
ax2.set_xlabel('Unit (sorted by CATE)')
ax2.set_ylabel('Outcome / CATE ($)')
ax2.set_title('Predicted CATEs vs Observed Outcomes (sorted)')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 6. Analyze Treatment Effects by Subgroups

In [ ]:
# Add predictions back to dataframe using saved indices
df_test = df.iloc[test_indices].copy()
df_test['cate_pred'] = cate_pred

# Analyze by subgroup
print("CATE by Education Level:")
subgroup_analysis = df_test.groupby('nodegree').agg({
    'cate_pred': ['mean', 'std', 'count'],
    're78': 'mean'
}).round(2)
print(subgroup_analysis)

print("\nCATE by Race:")
subgroup_race = df_test.groupby('black').agg({
    'cate_pred': ['mean', 'std', 'count'],
    're78': 'mean'
}).round(2)
print(subgroup_race)

# Plot subgroup effects
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# By education
edu_data = [df_test[df_test['nodegree']==0]['cate_pred'].values,
            df_test[df_test['nodegree']==1]['cate_pred'].values]
axes[0].boxplot(edu_data, labels=['Has Degree', 'No Degree'])
axes[0].axhline(y=0, color='red', linestyle='--')
axes[0].set_ylabel('CATE ($)')
axes[0].set_title('CATE by Education')
axes[0].grid(alpha=0.3)

# By prior earnings quartile
df_test['re74_quartile'] = pd.qcut(df_test['re74'], q=4, duplicates='drop')
n_quartiles = df_test['re74_quartile'].nunique()
q_labels = [f'Q{i+1}' for i in range(n_quartiles)]
q_data = [df_test[df_test['re74_quartile']==q]['cate_pred'].dropna().values
          for q in sorted(df_test['re74_quartile'].cat.categories)]
axes[1].boxplot(q_data, labels=q_labels)
axes[1].axhline(y=0, color='red', linestyle='--')
axes[1].set_ylabel('CATE ($)')
axes[1].set_title('CATE by Prior Earnings (1974)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Interpreting Your Results

| Output | What it means | What to look for |
|---|---|---|
| **ATE** | Average treatment effect across all individuals | Positive = job training raises earnings on average |
| **CATE by education** | Effect for high-school vs college graduates | Is the program more effective for the less educated? |
| **CATE by prior earnings** | Effect for low vs high earners | Diminishing returns? |
| **Causal tree splits** | Which covariates drive heterogeneity | Education and earnings are typically the most important |

**Key question**: Who benefits most from job training? Use the causal tree leaves to identify subgroups with the largest CATEs.

## Summary

This notebook replicates heterogeneous treatment effects analysis from Wager & Athey (2018):

1. **ATE**: ~$1,800 positive effect of job training
2. **Heterogeneity**: Effects vary substantially across individuals
3. **Key subgroups**: Those with no degree and lower prior earnings benefit more

**Extensions to try**:
- Implement honest causal forests (train/tree-building split)
- Compare adaptive vs honest estimation
- Use SHAP values for feature importance
- Test robustness to tree depth